# Build the Neural Network

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Get Device for Training

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = "cpu"
print(f"Using {device} device")

Using cuda device


## Define the Class

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([0], device='cuda:0')


## Model Layers

In [6]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.3350,  0.5348,  0.1287, -0.2379,  0.1445,  0.3117,  0.5229, -0.0020,
         -0.3418, -0.5610,  0.3791, -0.2999, -0.3001,  0.0786,  0.2165, -0.4572,
         -0.1599,  0.4158, -0.2449, -0.4787],
        [ 0.0159,  0.3158,  0.2796, -0.4290,  0.1594,  0.2515,  0.5050, -0.0951,
         -0.1060, -0.5087,  0.1880, -0.5375, -0.0754,  0.0817,  0.2117, -0.2211,
          0.1152,  0.7306, -0.2754, -0.7327],
        [-0.2404,  0.5338,  0.4483, -0.2555, -0.3280,  0.2256,  0.5663, -0.3803,
         -0.1330, -0.4497,  0.4856, -0.7316, -0.2033, -0.1198,  0.4901, -0.2415,
          0.0424,  0.3677, -0.2158, -0.3382]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.5348, 0.1287, 0.0000, 0.1445, 0.3117, 0.5229, 0.0000, 0.0000,
         0.0000, 0.3791, 0.0000, 0.0000, 0.0786, 0.2165, 0.0000, 0.0000, 0.4158,
         0.0000, 0.0000],
        [0.0159, 0.3158, 0.2796, 0.0000, 0.1594, 0.2515, 0.5050, 0.0000, 0.0000,
         0.0000, 0.1880, 0.0000, 0.0000, 0.0817, 0.21

In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

In [11]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

## Model Parameters

In [12]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0181, -0.0281, -0.0081,  ...,  0.0176,  0.0226,  0.0313],
        [ 0.0192, -0.0054, -0.0338,  ..., -0.0063, -0.0155,  0.0124]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0248,  0.0004], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0365, -0.0390,  0.0092,  ..., -0.0291,  0.0327, -0.0349],
        [ 0.0425,  0.0269,  0.0204,  ..., -0.0267,  0.0029, -0.0227]],
       device='cuda:0', grad_fn=<Slice